# Run Pipeline Evaluation & Analysis

Analysis of the outputs of `doc-grader` Tool against the established Human reference dataset, and plots the results.

Produces the `eval_gold_per_student.csv` and `eval_gold_summary.json` artefacts inside `outputs/gold_eval`, which are then loaded for visualisation in the subsequent cells.

## 1. Environment and Imports

In [ ]:
import logging
from pathlib import Path

import pandas as pd
from IPython.display import display
from scripts import eval
from scripts import eval_analysis as ea
from scripts import visual_utils as vu

from doc_grader.utils import configure_logging

configure_logging(logging.INFO)
logger = logging.getLogger("notebooks.evaluation_thesis_primary")
logging.getLogger("scripts.visual_utils").setLevel(logging.ERROR)
logging.getLogger("matplotlib").setLevel(logging.WARNING)

vu.configure_plot_style()
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

## 2. Paths and Evaluation Run

Set `RUN_EVALUATION` to `True` only when you want to rebuild `outputs/gold_eval_primary` from raw tool outputs.

In [ ]:
gold_csv = Path("../data/judge_gold/gold_ipp_data.csv")

variant_dirs = {
    "par": Path("../data/judge_gold/toolout/par"),
    "int": Path("../data/judge_gold/toolout_full/int"),
    "int_normal_judge_mini_content": Path("../data/judge_gold/toolout_full/int_normal_judge_mini_content"),
    "int_mini_judge_nano_content": Path("../data/judge_gold/toolout_full/int_mini_judge_nano_content"),
    "int_mini_judge_nano_content_nolocal": Path("../data/judge_gold/toolout_full/int_mini_judge_nano_content_nolocal"),
}

RUN_EVALUATION = False
PRIMARY_OUT_DIR = Path("../outputs/gold_eval_primary")
FALLBACK_OUT_DIR = Path("../outputs/gold_eval")
OUT_FIG_DIR = Path("../docs/eval_figs/")
BENCHMARK_DIR = Path("../outputs/docling_parser_benchmark")

OUT_FIG_DIR.mkdir(parents=True, exist_ok=True)

if RUN_EVALUATION:
    OUT_DIR = PRIMARY_OUT_DIR
    OUT_DIR.mkdir(parents=True, exist_ok=True)
else:
    primary_csv = PRIMARY_OUT_DIR / "eval_gold_per_student.csv"
    OUT_DIR = PRIMARY_OUT_DIR if primary_csv.exists() else FALLBACK_OUT_DIR

logger.info("Gold CSV: %s", gold_csv)
logger.info("Output dir: %s", OUT_DIR)
logger.info("Figure dir: %s", OUT_FIG_DIR)
logger.info("Benchmark dir: %s", BENCHMARK_DIR)

if RUN_EVALUATION:
    eval.evaluate_pipeline(
        variant_dirs={k: v if v.exists() else None for k, v in variant_dirs.items()},
        gold_path=gold_csv if gold_csv.exists() else None,
        out_dir=OUT_DIR,
        variant="all",
    )
else:
    logger.info("Skipping evaluate_pipeline; loading existing outputs only")

## 3. Load Data and Derive Core Metrics

In [ ]:
per_student_df, summary = ea.load_eval_data(eval_dir=OUT_DIR)

if per_student_df.empty:
    raise RuntimeError("No per-student rows loaded. Run evaluation or check OUT_DIR.")

per_student_df = ea.prepare_per_student_metrics(per_student_df)

logger.info("Loaded %d rows", len(per_student_df))
display(per_student_df.head(3))

## 4. Dataset Overview and Characteristics

Primary cohort facts and telemetry coverage checks used to scope all claims.

In [ ]:
gold_df = pd.read_csv(gold_csv)
student_variant_pairs = gold_df[["id", "task_variant"]].drop_duplicates()

overview_rows = [
    {"metric": "gold_rows", "value": len(gold_df)},
    {"metric": "unique_student_variant_pairs", "value": len(student_variant_pairs)},
    {"metric": "unique_students", "value": int(student_variant_pairs["id"].nunique())},
    {"metric": "gold_pairs_par", "value": int((student_variant_pairs["task_variant"] == "par").sum())},
    {"metric": "gold_pairs_int", "value": int((student_variant_pairs["task_variant"] == "int").sum())},
    {"metric": "eval_rows_total", "value": len(per_student_df)},
    {"metric": "eval_rows_par", "value": int((per_student_df["task_variant"] == "par").sum())},
    {"metric": "eval_rows_int", "value": int((per_student_df["task_variant"] == "int").sum())},
    {"metric": "int_rows_with_cost", "value": int(per_student_df.loc[per_student_df["task_variant"] == "int", "cost_eur"].notna().sum())},
    {"metric": "par_rows_with_cost", "value": int(per_student_df.loc[per_student_df["task_variant"] == "par", "cost_eur"].notna().sum())},
]
overview_df = pd.DataFrame(overview_rows)
display(overview_df)

cohort_split_table = (
    student_variant_pairs["task_variant"]
    .value_counts()
    .rename_axis("task_variant")
    .reset_index(name="n_students")
    .sort_values("task_variant")
    .reset_index(drop=True)
)
display(cohort_split_table)

doc_distribution_table = (
    gold_df[["id", "task_variant", "doc_type"]]
    .drop_duplicates()
    ["doc_type"]
    .value_counts()
    .rename_axis("doc_type")
    .reset_index(name="n_students")
    .sort_values("doc_type")
    .reset_index(drop=True)
)
display(doc_distribution_table)

## 5. Quality Core: Score Agreement and Student-Level Metrics

In [ ]:
ea.visualise_score_scatter(
    per_student_df,
    save_path=vu.build_figure_path("fig_6_3_score_scatter", OUT_FIG_DIR),
)

In [ ]:
quality_view = per_student_df[["task_variant", "directory_alias", "student_precision", "student_recall", "abs_points_delta_pct"]].copy()
display(quality_view.describe(include="all"))

ea.visualise_student_precision_by_variant(
    quality_view,
    save_path=vu.build_figure_path("fig_6_4_student_precision", OUT_FIG_DIR),
)
ea.visualise_student_recall_by_variant(
    quality_view,
    save_path=vu.build_figure_path("fig_6_5_student_recall", OUT_FIG_DIR),
)
ea.visualise_absolute_score_error_by_variant(
    quality_view,
    save_path=vu.build_figure_path("fig_6_6_absolute_score_error", OUT_FIG_DIR),
)

### MAE by Score Quartile

Mean Absolute Error sliced by the actual document score. This shows if the tool struggles more on high-scoring or low-scoring documents.

In [ ]:
ea.visualise_mae_by_score_quartile(per_student_df)

### Format Comparison

Compares PDF vs Markdown only on cost and latency.

By default this chart is filtered to the baseline full-eval aliases (`par`, `int`).
Use the scope table below to see exactly which model combinations are included in each alias.

In [ ]:
ea.visualise_format_comparison(per_student_df)

## 6. Quality by Code and Model

Per-code precision/recall uses the summary report. Model-level views use per-student aggregations with sample-size filtering.

In [ ]:
per_code_df = pd.DataFrame.from_dict(summary.get("per_code", {}), orient="index").reset_index(names="code")
if per_code_df.empty:
    raise RuntimeError("summary.per_code is empty")

per_code_df = per_code_df.sort_values("total_in_gold", ascending=False).head(15)
display(per_code_df[["code", "total_in_gold", "precision", "recall"]])

ea.visualise_per_code_precision_recall(
    summary,
    save_path=vu.build_figure_path("fig_6_7_per_code_precision_recall", OUT_FIG_DIR),
)



In [ ]:
model_quality_df = ea.summarise_group_quality(per_student_df, "generator_models", min_n=5)
display(model_quality_df)

ea.visualise_group_quality_bars(
    model_quality_df,
    title="Quality by Grader Model",
    label_formatter=lambda row: f"{row['group']} (n={int(row['n'])})",
    save_path=vu.build_figure_path("fig_6_8_quality_by_grader_model", OUT_FIG_DIR),
)

In [ ]:
def format_pair_label(row: pd.Series) -> str:
    if " | " in row["group"]:
        grader, judge = row["group"].split(" | ", 1)
        return f"Grader: {grader}\nJudge: {judge}\n(n={int(row['n'])})"
    return f"{row['group']}\n(n={int(row['n'])})"


pair_quality_df = ea.summarise_group_quality(per_student_df, "grader_judge_pair", min_n=5).head(10)
display(pair_quality_df)

ea.visualise_group_quality_bars(
    pair_quality_df,
    title="Quality by Grader + Judge Pair (Top 10 by N)",
    label_formatter=format_pair_label,
    save_path=vu.build_figure_path("fig_6_9_quality_by_grader_judge_pair", OUT_FIG_DIR),
)

### Code Frequency Comparison

How often Tool applied a code vs how often Human did.

In [ ]:
ea.visualise_code_frequency_comparison(summary)

### Per-Code Agreement

Agreement percentage by code.

In [ ]:
ea.visualise_per_code_agreement(summary)

### Precision vs Recall Bubble Chart

How well the agent identifies specific deduction codes. Bubble size indicates how common the code is in the dataset.

In [ ]:
ea.visualise_precision_recall_bubble(summary)

### Impact Delta Scatter

How far off the point deductions were for individual codes.

In [ ]:
ea.visualise_impact_delta_scatter(per_student_df)

## 7. Pipeline Performance & Filtering

### Pipeline Waterfall

Shows how many putative findings enter the judge stage and how many survive to become final codes.

In [ ]:
ea.visualise_pipeline_waterfall(summary)

### Judge Survival Rate

In [ ]:
ea.visualise_judge_survival_rate(summary)

## 8. Operational Metrics (INT Only)

These charts are explicitly scoped to rows with non-null operational telemetry.
The section ends with one benchmark carry-over figure that shows how Docling PDF parse time changes across parser configurations.

In [ ]:
ops_df = per_student_df[per_student_df["cost_eur"].notna()].copy()
logger.info("Operational rows: %d", len(ops_df))
display(ops_df[["task_variant", "directory_alias", "generator_models", "judge_models", "cost_eur", "generator_cost_eur", "judge_cost_eur", "elapsed_seconds"]].head(5))

ea.visualise_operational_metric_by_model(
    ops_df,
    metric_key="generator_cost",
    save_path=vu.build_figure_path("fig_6_10_generator_cost_by_grader_model", OUT_FIG_DIR),
)
ea.visualise_operational_metric_by_model(
    ops_df,
    metric_key="judge_cost",
    save_path=vu.build_figure_path("fig_6_11_judge_cost_by_judge_model", OUT_FIG_DIR),
)
ea.visualise_operational_metric_by_model(
    ops_df,
    metric_key="latency",
    save_path=vu.build_figure_path("fig_6_12_latency_by_grader_model", OUT_FIG_DIR),
)

In [ ]:
stage_df = ops_df.dropna(subset=["parse_time", "analyser_time", "judge_time"]).copy()
stage_summary = ea.summarise_stage_time_composition(stage_df)
display(stage_summary)

ea.visualise_stage_time_composition(
    stage_df,
    save_path=vu.build_figure_path("fig_6_13_stage_time_composition", OUT_FIG_DIR),
)
ea.visualise_stage_times(
    stage_df,
    save_path=vu.build_figure_path("fig_6_14_pipeline_stage_latency", OUT_FIG_DIR),
)

In [ ]:
benchmark_df = ea.load_docling_parser_benchmark(BENCHMARK_DIR)
benchmark_pdf_summary = (
    benchmark_df.loc[benchmark_df["doc_type"] == "pdf", ["mode", "elapsed_seconds"]]
    .groupby("mode", observed=True)
    .agg(
        n_documents=("elapsed_seconds", "size"),
        mean_seconds=("elapsed_seconds", "mean"),
        median_seconds=("elapsed_seconds", "median"),
        p95_seconds=("elapsed_seconds", lambda values: float(values.quantile(0.95))),
    )
    .reset_index()
 )
display(benchmark_pdf_summary)

ea.visualise_docling_parser_timing_by_mode(
    benchmark_df,
    doc_type="pdf",
    save_path=vu.build_figure_path("fig_6_15_docling_pdf_parse_time_by_config", OUT_FIG_DIR),
)

### Cost by Task Variant

In [ ]:
ea.visualise_cost_by_variant(per_student_df)

### Cost vs Document Score

In [ ]:
ea.visualise_cost_vs_doc_points(per_student_df)

### Mean Token Usage by Task Variant (Table)

Token usage is shown as a table instead of a chart.

In [ ]:
token_usage_df = ea.summarise_token_usage_by_variant(per_student_df)
display(token_usage_df)

### Latency by Variant

In [ ]:
ea.visualise_latency_by_variant(per_student_df)

## 9. Aggregated Summaries

### Metrics per Variant Summary

In [ ]:
ea.visualise_per_variant_metrics(summary)

### Metrics per Format Summary

Overall performance metrics split by document format (PDF vs Markdown).

In [ ]:
ea.visualise_per_format_metrics(summary)

### Metrics per Language Summary

Overall performance metrics split by the student's document language.

In [ ]:
ea.visualise_per_language_metrics(summary)

### Metrics per Generator Model

Overall performance metrics split by the content/asset AI model used.

In [ ]:
ea.visualise_per_generator_model_metrics(summary)

### Metrics per Judge Model

Overall performance metrics split by the judge AI model used.

In [ ]:
ea.visualise_per_judge_model_metrics(summary)

In [ ]:
variant_summary_data = []

for variant, stats in summary.get("per_variant", {}).items():
    variant_summary_data.append({
        "Variant": variant.upper(),
        "Rows": stats.get("n"),
        "Mean Precision": stats.get("macro_precision"),
        "Mean Recall": stats.get("macro_recall"),
        "MAE [pp]": stats.get("points_mae_pct")
    })

variant_summary_df = pd.DataFrame(variant_summary_data)
display(variant_summary_df)

In [ ]:
median_summary = quality_view.groupby("task_variant", observed=True)[
    ["student_precision", "student_recall", "abs_points_delta_pct"]
].median().reset_index()

median_summary.columns = ["Variant", "Med. Prec.", "Med. Rec.", "MedAE [pp]"]
median_summary["Variant"] = median_summary["Variant"].str.upper()

display(median_summary)